In [2]:
# Load the raw CSV file from the Bronze folder

bronze_file_path = "Files/bronze/uk_supermarket_supply_2000.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_file_path)
)

print("Number of rows:", df_raw.count())
print("Number of columns:", len(df_raw.columns))

df_raw.printSchema()
display(df_raw.limit(20))








StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 4, Finished, Available, Finished, False)

Number of rows: 2000
Number of columns: 11
root
 |-- Date: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Opening_Stock: integer (nullable = true)
 |-- Deliveries: integer (nullable = true)
 |-- Daily_Sales: integer (nullable = true)
 |-- Closing_Stock: integer (nullable = true)
 |-- Lead_Time_Days: integer (nullable = true)
 |-- Supplier_Status: string (nullable = true)
 |-- Stock_Status: string (nullable = true)



SynapseWidget(Synapse.DataFrame, ce7c3de6-5d22-4c67-a41d-5bfde6ded4b9)

In [3]:
# Import PySpark functions for data-quality calculations
from pyspark.sql import functions as F

# Count null values in every column
null_counts = df_raw.select(
    [
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in df_raw.columns
    ]
)

# Count exact duplicate rows
duplicate_count = df_raw.count() - df_raw.dropDuplicates().count()

# Find mathematically impossible inventory records
df_impossible_sales = df_raw.filter(
    F.col("Daily_Sales")
    > (F.col("Opening_Stock") + F.col("Deliveries"))
)

# Display the main data-quality results
print("Number of exact duplicate rows:", duplicate_count)
print("Number of impossible inventory rows:", df_impossible_sales.count())

# Display null counts and impossible records
display(null_counts)
display(df_impossible_sales.limit(20))

StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 5, Finished, Available, Finished, False)

Number of exact duplicate rows: 0
Number of impossible inventory rows: 44


SynapseWidget(Synapse.DataFrame, ae06a5de-aa0d-4b01-b626-dbf05ebb4e51)

SynapseWidget(Synapse.DataFrame, 2e6e6561-a501-47ce-9ac1-7d2c1252849c)

In [4]:
# Convert the Date column to a proper date and enforce numeric data types

numeric_columns = [
    "Opening_Stock",
    "Deliveries",
    "Daily_Sales",
    "Closing_Stock",
    "Lead_Time_Days"
]

df_transformed = df_raw.withColumn(
    "Date",
    F.to_date(F.col("Date"), "dd/MM/yyyy")
)

for column_name in numeric_columns:
    df_transformed = df_transformed.withColumn(
        column_name,
        F.col(column_name).cast("int")
    )

df_transformed.printSchema()
display(df_transformed.limit(20))

StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 6, Finished, Available, Finished, False)

root
 |-- Date: date (nullable = true)
 |-- Region: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Opening_Stock: integer (nullable = true)
 |-- Deliveries: integer (nullable = true)
 |-- Daily_Sales: integer (nullable = true)
 |-- Closing_Stock: integer (nullable = true)
 |-- Lead_Time_Days: integer (nullable = true)
 |-- Supplier_Status: string (nullable = true)
 |-- Stock_Status: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 29df7ad4-8e38-4113-8696-32b1e6a68a55)

In [5]:
# Identify mathematically impossible inventory records

impossible_sales_condition = (
    F.col("Daily_Sales")
    > (F.col("Opening_Stock") + F.col("Deliveries"))
)

# Separate corrupted records for quarantine
df_quarantined = df_transformed.filter(
    impossible_sales_condition
)

# Keep only valid records for further processing
df_validated = df_transformed.filter(
    ~impossible_sales_condition
)

print("Original number of rows:", df_transformed.count())
print("Quarantined rows:", df_quarantined.count())
print("Valid rows:", df_validated.count())

display(df_quarantined.limit(20))

StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 7, Finished, Available, Finished, False)

Original number of rows: 2000
Quarantined rows: 44
Valid rows: 1956


SynapseWidget(Synapse.DataFrame, 323ee2bd-18ea-4f33-928e-32e2d84b68a6)

In [6]:
# Save the corrupted records as quarantined_data.csv

temporary_quarantine_path = "Files/quarantine/quarantined_data_temp"
final_quarantine_path = "Files/quarantine/quarantined_data.csv"

# Write the quarantined records temporarily as one CSV part file
(
    df_quarantined
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(temporary_quarantine_path)
)

# Find the CSV part file created by Spark
quarantine_csv_file = [
    file.path
    for file in notebookutils.fs.ls(temporary_quarantine_path)
    if file.name.endswith(".csv")
][0]

# Remove an older final file if it already exists
if notebookutils.fs.exists(final_quarantine_path):
    notebookutils.fs.rm(final_quarantine_path)

# Rename the Spark part file to quarantined_data.csv
notebookutils.fs.mv(
    quarantine_csv_file,
    final_quarantine_path
)

# Remove the temporary folder
notebookutils.fs.rm(
    temporary_quarantine_path,
    True
)

print("Quarantine file created successfully.")
print("File location:", final_quarantine_path)
print("Number of quarantined rows:", df_quarantined.count())

StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 8, Finished, Available, Finished, False)

Quarantine file created successfully.
File location: Files/quarantine/quarantined_data.csv
Number of quarantined rows: 44


In [7]:
# Preserve the original Stock_Status for the later False Alarm Audit
# and recalculate the correct Stock_Status

df_status_corrected = (
    df_validated
    .withColumn(
        "Original_Stock_Status",
        F.col("Stock_Status")
    )
    .withColumn(
        "Stock_Status",
        F.when(
            F.col("Closing_Stock") < 50,
            F.lit("Critical")
        )
        .when(
            F.col("Closing_Stock") < 150,
            F.lit("Warning")
        )
        .otherwise(
            F.lit("Healthy")
        )
    )
)

# Count how many incorrect status labels were corrected
corrected_status_count = df_status_corrected.filter(
    F.col("Original_Stock_Status") != F.col("Stock_Status")
).count()

print("Number of Stock_Status labels corrected:", corrected_status_count)

display(
    df_status_corrected.select(
        "Date",
        "Store_ID",
        "Closing_Stock",
        "Original_Stock_Status",
        "Stock_Status"
    ).limit(20)
)

StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 9, Finished, Available, Finished, False)

Number of Stock_Status labels corrected: 64


SynapseWidget(Synapse.DataFrame, c9dfa4d3-8545-4fc9-bfc1-23a6f985c9ee)

In [8]:
# Import the integer data type for the reliability score
from pyspark.sql.types import IntegerType

# Create the supplier reliability scoring function
def calculate_supplier_reliability(supplier_status, lead_time_days):
    score = 100

    if supplier_status == "On-Time":
        score += 10
    elif supplier_status == "Delayed":
        score -= 30
    elif supplier_status == "Cancelled":
        score -= 60

    score -= 2 * lead_time_days

    return max(score, 0)

# Convert the Python function into a PySpark UDF
supplier_reliability_udf = F.udf(
    calculate_supplier_reliability,
    IntegerType()
)

# Apply the function to every valid row
df_reliability_scored = df_status_corrected.withColumn(
    "Supplier_Reliability_Score",
    supplier_reliability_udf(
        F.col("Supplier_Status"),
        F.col("Lead_Time_Days")
    )
)

display(
    df_reliability_scored.select(
        "Date",
        "Store_ID",
        "Supplier_Status",
        "Lead_Time_Days",
        "Supplier_Reliability_Score"
    ).limit(20)
)

StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bcf00265-6719-4c72-b7d5-8aca1c2fdc72)

In [9]:
# Calculate the Category Volatility Index

category_volatility = (
    df_reliability_scored
    .groupBy("Category")
    .agg(
        F.avg("Daily_Sales").alias("Mean_Daily_Sales"),
        F.stddev_samp("Daily_Sales").alias("StdDev_Daily_Sales")
    )
    .withColumn(
        "Category_Volatility_Index",
        (
            F.col("StdDev_Daily_Sales")
            / F.col("Mean_Daily_Sales")
        ) * 100
    )
)

# Map the Category Volatility Index back to the main DataFrame
df_enriched = df_reliability_scored.join(
    category_volatility.select(
        "Category",
        "Category_Volatility_Index"
    ),
    on="Category",
    how="left"
)

display(
    category_volatility.select(
        "Category",
        "Mean_Daily_Sales",
        "StdDev_Daily_Sales",
        "Category_Volatility_Index"
    ).orderBy("Category")
)

display(
    df_enriched.select(
        "Date",
        "Store_ID",
        "Category",
        "Daily_Sales",
        "Supplier_Reliability_Score",
        "Category_Volatility_Index"
    ).limit(20)
)


StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0c5af4db-d1b2-452b-a87e-8baf9beab9e8)

SynapseWidget(Synapse.DataFrame, 1b3c7e50-93ff-4aef-8d52-d79dcdedfe2c)

In [10]:
# Load the validated and enriched dataset into the Lakehouse SQL table

table_name = "uk_supermarket_supply"

(
    df_enriched
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print("Table created successfully:", table_name)
print("Number of rows loaded:", spark.table(table_name).count())

display(
    spark.table(table_name).limit(20)
)

StatementMeta(, d9f5f806-41eb-407c-9d48-f4fc69459b6f, 12, Finished, Available, Finished, False)

Table created successfully: uk_supermarket_supply
Number of rows loaded: 1956


SynapseWidget(Synapse.DataFrame, 0cc09aae-7523-4fb0-ba0f-0e7b0b523453)